# NHL Totals Betting Capstone — MoneyPuck-Only Pipeline
**Goal:** Build a reproducible, end-to-end dataset + modeling workflow for predicting **NHL game total goals** using **MoneyPuck** only.

This notebook:
1. Downloads MoneyPuck `all_teams.csv` (cached locally)
2. Filters to Team Level / All situations
3. Engineers rolling + EWM + trend + strength-of-schedule features
4. Pivots to game-level (home/away) and creates the target `total_goals`
5. Cleans the master (drops constant/duplicate columns)
6. Produces the 5 EDA figures used in the final report
7. Trains & evaluates:
   - Linear Regression baseline (time-aware split + TimeSeries CV + diagnostics)
   - Tuned XGBoost (time-aware split + TimeSeries CV + diagnostics + feature importances)
8. Unsupervised: team-style clustering (k=2) with PCA + cluster mean heatmap + silhouette scores

Outputs are written to:
- `data/master_cleaned.csv`
- `figures/*.png`
- `artifacts/metrics.json` and `artifacts/team_clusters_k2.csv`

In [2]:
# ============================================================
# 0) Imports + Reproducibility + Folders
# ============================================================
from __future__ import annotations

import gc
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan

from xgboost import XGBRegressor

RANDOM_SEED = 21
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data")
FIG_DIR = Path("figures")
ART_DIR = Path("artifacts")

DATA_DIR.mkdir(exist_ok=True, parents=True)
FIG_DIR.mkdir(exist_ok=True, parents=True)
ART_DIR.mkdir(exist_ok=True, parents=True)

print("DATA_DIR:", DATA_DIR.resolve())
print("FIG_DIR :", FIG_DIR.resolve())
print("ART_DIR :", ART_DIR.resolve())


DATA_DIR: /Users/thegootch/Desktop/Data Science/merrimack/capstone/capstone_project/data
FIG_DIR : /Users/thegootch/Desktop/Data Science/merrimack/capstone/capstone_project/figures
ART_DIR : /Users/thegootch/Desktop/Data Science/merrimack/capstone/capstone_project/artifacts


## 1) Download MoneyPuck data (cached)
MoneyPuck provides team-level game-by-game stats. We cache the CSV locally so re-runs are fast and reproducible.

In [4]:
# ============================================================
# 1) Download MoneyPuck (cached)
# ============================================================
MONEYPUCK_URL = "https://moneypuck.com/moneypuck/playerData/careers/gameByGame/all_teams.csv"
MONEYPUCK_RAW_FILE = DATA_DIR / "moneypuck_all_teams_raw.csv"

def download_moneypuck(force: bool = False) -> Path:
    """
    Download MoneyPuck all_teams.csv if it doesn't exist locally or if force=True.
    Returns the local filepath.
    """
    if MONEYPUCK_RAW_FILE.exists() and not force:
        print(f"Using cached MoneyPuck file: {MONEYPUCK_RAW_FILE}")
        return MONEYPUCK_RAW_FILE

    print("Downloading MoneyPuck all_teams.csv ...")
    df_raw = pd.read_csv(MONEYPUCK_URL)
    df_raw.to_csv(MONEYPUCK_RAW_FILE, index=False)
    print(f"Saved MoneyPuck data to {MONEYPUCK_RAW_FILE} (rows={len(df_raw):,}, cols={df_raw.shape[1]:,})")
    return MONEYPUCK_RAW_FILE

# Set force=True to re-download
download_moneypuck(force=False)


Using cached MoneyPuck file: data/moneypuck_all_teams_raw.csv


PosixPath('data/moneypuck_all_teams_raw.csv')

## 2) Load and filter to Team Level / All situations
We only keep **Team Level** rows and **All situations** to ensure we’re modeling team performance, not individual players or special teams slices.

In [6]:
# ============================================================
# 2) Load + filter team-level, all situations
# ============================================================
mp = pd.read_csv(MONEYPUCK_RAW_FILE)

# MoneyPuck gameDate is typically YYYYMMDD as an integer
mp["gameDate"] = pd.to_datetime(mp["gameDate"].astype(str), format="%Y%m%d", errors="coerce")

mp_team = mp[(mp["position"] == "Team Level") & (mp["situation"].astype(str).str.lower() == "all")].copy()

print("Filtered Team Level / All situations:", mp_team.shape)

# Core identifiers
id_cols = [
    "gameId", "gameDate", "season", "team", "playerTeam", "opposingTeam",
    "home_or_away", "playoffGame"
]

# Stat columns we keep 
KEEP_STAT_COLS = [
    "goalsFor", "goalsAgainst",
    "shotsOnGoalFor", "shotsOnGoalAgainst",
    "shotAttemptsFor", "shotAttemptsAgainst",
    "blockedShotAttemptsFor", "blockedShotAttemptsAgainst",
    "missedShotsFor", "missedShotsAgainst",
    "highDangerShotsFor", "highDangerShotsAgainst",
    "mediumDangerShotsFor", "mediumDangerShotsAgainst",
    "lowDangerShotsFor", "lowDangerShotsAgainst",
    "totalShotCreditFor", "totalShotCreditAgainst",
    "scoreAdjustedTotalShotCreditFor", "scoreAdjustedTotalShotCreditAgainst",
    "xGoalsFor", "xGoalsAgainst",
    "flurryAdjustedxGoalsFor", "flurryAdjustedxGoalsAgainst",
    "scoreVenueAdjustedxGoalsFor", "scoreVenueAdjustedxGoalsAgainst",
    "xGoalsPercentage", "corsiPercentage", "fenwickPercentage",
    "lowDangerxGoalsFor", "lowDangerxGoalsAgainst",
    "mediumDangerxGoalsFor", "mediumDangerxGoalsAgainst",
    "highDangerxGoalsFor", "highDangerxGoalsAgainst",
    "penaltiesFor", "penaltiesAgainst",
    "penalityMinutesFor", "penalityMinutesAgainst",
    "hitsFor", "hitsAgainst",
    "takeawaysFor", "takeawaysAgainst",
    "giveawaysFor", "giveawaysAgainst",
    "faceOffsWonFor", "faceOffsWonAgainst",
]

available_stats = [c for c in KEEP_STAT_COLS if c in mp_team.columns]
mp_team = mp_team[id_cols + available_stats].copy()

# Standardize names
mp_team = mp_team.rename(columns={
    "gameId":        "game_id",
    "gameDate":      "game_date",
    "team":          "team_code",
    "playerTeam":    "team_code_dup",
    "opposingTeam":  "opp_code",
    "home_or_away":  "home_away",
})

mp_team["game_date"] = pd.to_datetime(mp_team["game_date"])
mp_team["is_home"] = (mp_team["home_away"].astype(str).str.upper() == "HOME").astype(int)
mp_team["is_playoff"] = mp_team["playoffGame"].fillna(0).astype(int)

# Per-game derived metrics
mp_team["goal_diff"] = mp_team["goalsFor"] - mp_team["goalsAgainst"]
mp_team["win"] = (mp_team["goalsFor"] > mp_team["goalsAgainst"]).astype(int)

# Ensure share metrics are floats
for c in ["xGoalsPercentage", "corsiPercentage", "fenwickPercentage"]:
    if c in mp_team.columns:
        mp_team[c] = mp_team[c].astype(float)

mp_team.head()


Filtered Team Level / All situations: (44466, 111)


,game_id,game_date,season,team_code,team_code_dup,opp_code,home_away,playoffGame,goalsFor,goalsAgainst,...,takeawaysFor,takeawaysAgainst,giveawaysFor,giveawaysAgainst,faceOffsWonFor,faceOffsWonAgainst,is_home,is_playoff,goal_diff,win
1,2008020001,2008-10-04,2008,NYR,NYR,T.B,AWAY,0,2.0,1.0,...,19.0,11.0,8.0,7.0,30.0,32.0,0,0,1.0,1
6,2008020003,2008-10-05,2008,NYR,NYR,T.B,HOME,0,2.0,1.0,...,12.0,8.0,11.0,7.0,17.0,29.0,1,0,1.0,1
11,2008020010,2008-10-10,2008,NYR,NYR,CHI,HOME,0,4.0,2.0,...,7.0,8.0,6.0,5.0,23.0,32.0,1,0,2.0,1
16,2008020019,2008-10-11,2008,NYR,NYR,PHI,AWAY,0,4.0,3.0,...,4.0,5.0,3.0,8.0,39.0,29.0,0,0,1.0,1
21,2008020034,2008-10-13,2008,NYR,NYR,N.J,HOME,0,4.0,1.0,...,13.0,9.0,10.0,3.0,28.0,20.0,1,0,3.0,1


## 3) Feature engineering: rolling windows, EWMs, and trend features
Build pre-game form / style using rolling means and EWMs for each team, then derive trend features (roll7 − roll30) for share metrics.

In [8]:
# ============================================================
# 3) Rolling + EWM features per team
# ============================================================
ROLL_WINDOWS = [3, 7, 15, 30]
EWM_ALPHAS = {"fast": 0.3, "slow": 0.1}  # fast emphasizes recent games; slow captures longer form

metrics_for_roll = [
    "goalsFor", "goalsAgainst", "goal_diff", "win",
    "xGoalsFor", "xGoalsAgainst",
    "shotsOnGoalFor", "shotsOnGoalAgainst",
    "shotAttemptsFor", "shotAttemptsAgainst",
    "highDangerShotsFor", "highDangerShotsAgainst",
    "totalShotCreditFor", "totalShotCreditAgainst",
    "xGoalsPercentage", "corsiPercentage", "fenwickPercentage",
]
metrics_for_roll = [m for m in metrics_for_roll if m in mp_team.columns]

def add_team_rollings(df_team: pd.DataFrame) -> pd.DataFrame:
    df_team = df_team.sort_values("game_date").copy()
    df_team["games_played"] = np.arange(1, len(df_team) + 1)

    roll_frames = []

    for col in metrics_for_roll:
        s = df_team[col]
        sub = {}
        for w in ROLL_WINDOWS:
            sub[f"{col}_roll{w}"] = s.rolling(window=w, min_periods=1).mean()
        for label, alpha in EWM_ALPHAS.items():
            sub[f"{col}_ewm_{label}"] = s.ewm(alpha=alpha, adjust=False).mean()
        roll_frames.append(pd.DataFrame(sub, index=df_team.index))

    # win% rolling windows
    win_sub = {}
    for w in [5, 10, 20]:
        win_sub[f"win_pct_roll{w}"] = df_team["win"].rolling(window=w, min_periods=1).mean()
    roll_frames.append(pd.DataFrame(win_sub, index=df_team.index))

    # trend features: roll7 - roll30 for share metrics
    for base in ["xGoalsPercentage", "corsiPercentage", "fenwickPercentage"]:
        if f"{base}_roll7" in roll_frames[0].columns and f"{base}_roll30" in roll_frames[0].columns:
            pass

    roll_all = pd.concat(roll_frames, axis=1)

    for base in ["xGoalsPercentage", "corsiPercentage", "fenwickPercentage"]:
        short = f"{base}_roll7"
        long  = f"{base}_roll30"
        if short in roll_all.columns and long in roll_all.columns:
            roll_all[f"{base}_trend_7v30"] = roll_all[short] - roll_all[long]

    return pd.concat([df_team, roll_all], axis=1)

teams = []
for team_code, grp in mp_team.groupby("team_code"):
    teams.append(add_team_rollings(grp))

mp_team_roll = pd.concat(teams, ignore_index=True)
mp_team_roll = mp_team_roll.sort_values(["team_code", "game_date", "game_id"]).drop_duplicates(
    subset=["team_code", "game_date", "game_id"],
    keep="first"
)

print("Team-level with rolling/EWM shape:", mp_team_roll.shape)
mp_team_roll.head()


Team-level with rolling/EWM shape: (44466, 168)


,game_id,game_date,season,team_code,team_code_dup,opp_code,home_away,playoffGame,goalsFor,goalsAgainst,...,fenwickPercentage_roll15,fenwickPercentage_roll30,fenwickPercentage_ewm_fast,fenwickPercentage_ewm_slow,win_pct_roll5,win_pct_roll10,win_pct_roll20,xGoalsPercentage_trend_7v30,corsiPercentage_trend_7v30,fenwickPercentage_trend_7v30
0,2008020008,2008-10-09,2008,ANA,ANA,S.J,AWAY,0,1.0,4.0,...,0.384600,0.384600,0.384600,0.384600,0.0,0.0,0.0,0.0,0.0,0.0
1,2008020030,2008-10-12,2008,ANA,ANA,ARI,HOME,0,2.0,4.0,...,0.497200,0.497200,0.452160,0.407120,0.0,0.0,0.0,0.0,0.0,0.0
2,2008020042,2008-10-14,2008,ANA,ANA,L.A,AWAY,0,3.0,6.0,...,0.495867,0.495867,0.464472,0.415728,0.0,0.0,0.0,0.0,0.0,0.0
3,2008020048,2008-10-15,2008,ANA,ANA,EDM,HOME,0,2.0,3.0,...,0.508000,0.508000,0.488450,0.428595,0.0,0.0,0.0,0.0,0.0,0.0
4,2008020061,2008-10-17,2008,ANA,ANA,S.J,HOME,0,4.0,0.0,...,0.488220,0.488220,0.464645,0.426646,0.2,0.2,0.2,0.0,0.0,0.0


## 4) Strength-of-schedule (SoS) features
Estimate opponent strength using lagged xG% roll30, merge opponent strength onto each game, and then compute rolling SoS averages.

In [10]:
# ============================================================
# 4) Strength of schedule features (SoS)
# ============================================================
# Pre-game team strength: lagged xGoalsPercentage_roll30
if "xGoalsPercentage_roll30" in mp_team_roll.columns:
    mp_team_roll = mp_team_roll.sort_values(["team_code", "game_date"])
    mp_team_roll["team_strength_xg30"] = mp_team_roll.groupby("team_code")["xGoalsPercentage_roll30"].shift(1)
else:
    mp_team_roll["team_strength_xg30"] = np.nan

# Attach opponent strength per game
strength_ref = mp_team_roll[["game_id", "team_code", "team_strength_xg30"]].rename(columns={"team_code": "opp_code"})
mp_team_roll = mp_team_roll.merge(
    strength_ref,
    on=["game_id", "opp_code"],
    how="left",
    suffixes=("", "_opp")
)

# Rolling SoS per team
SOS_WINDOWS = [5, 10, 20]
if "team_strength_xg30_opp" in mp_team_roll.columns:
    mp_team_roll = mp_team_roll.sort_values(["team_code", "game_date"])
    for w in SOS_WINDOWS:
        mp_team_roll[f"sos_xg30_roll{w}"] = (
            mp_team_roll.groupby("team_code")["team_strength_xg30_opp"]
            .rolling(window=w, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )

mp_team_roll[["team_code","game_date","team_strength_xg30","team_strength_xg30_opp","sos_xg30_roll10"]].head()


,team_code,game_date,team_strength_xg30,team_strength_xg30_opp,sos_xg30_roll10
0,ANA,2008-10-09,NaN,NaN,NaN
1,ANA,2008-10-12,0.259400,0.35390,0.353900
2,ANA,2008-10-14,0.403750,0.40655,0.380225
3,ANA,2008-10-15,0.368433,0.36160,0.374017
4,ANA,2008-10-17,0.402525,0.60180,0.430962


## 5) Pivot to game-level + create target `total_goals`
Merge home and away rows into one game row, then compute:
- `home_goals`, `away_goals`
- `total_goals` (primary target)
- plus differential features `diff_*` (home − away) for numeric columns.

In [12]:
# ============================================================
# 5) Pivot to game-level + targets + diff features
# ============================================================
game_chunks = []
seasons = sorted(mp_team_roll["season"].dropna().unique())

for season in seasons:
    df_season = mp_team_roll[mp_team_roll["season"] == season].copy()

    home = df_season[df_season["is_home"] == 1].copy()
    away = df_season[df_season["is_home"] == 0].copy()

    merge_keys = ["game_id", "game_date", "season"]
    g = home.merge(away, on=merge_keys, suffixes=("_home", "_away"))

    # Canonical date
    g["game_date"] = pd.to_datetime(g["game_date"]).dt.normalize()

    # Targets
    g["home_goals"] = g["goalsFor_home"]
    g["away_goals"] = g["goalsFor_away"]
    g["home_win"] = (g["home_goals"] > g["away_goals"]).astype("int8")
    g["home_win_margin"] = (g["home_goals"] - g["away_goals"]).astype("int16")
    g["total_goals"] = g["home_goals"] + g["away_goals"]

    # Playoff flag
    if {"is_playoff_home", "is_playoff_away"}.issubset(g.columns):
        g["is_playoff"] = g[["is_playoff_home", "is_playoff_away"]].max(axis=1)

    # Diff features (home - away)
    all_cols = set(g.columns)
    home_cols = [c for c in all_cols if c.endswith("_home")]
    away_cols = [c for c in all_cols if c.endswith("_away")]
    base_names = set([c[:-5] for c in home_cols]) & set([c[:-5] for c in away_cols])

    diff_data = {}
    for base in base_names:
        h_col = f"{base}_home"
        a_col = f"{base}_away"
        if np.issubdtype(g[h_col].dtype, np.number) and np.issubdtype(g[a_col].dtype, np.number):
            diff_data[f"diff_{base}"] = g[h_col] - g[a_col]

    if diff_data:
        g = pd.concat([g, pd.DataFrame(diff_data, index=g.index)], axis=1)

    game_chunks.append(g)

    del df_season, home, away, g
    gc.collect()

game_df = pd.concat(game_chunks, ignore_index=True)
print("Game-level master shape:", game_df.shape)
game_df[["game_id","game_date","season","team_code_home","team_code_away","total_goals"]].head()


Game-level master shape: (22118, 515)


,game_id,game_date,season,team_code_home,team_code_away,total_goals
0,2008020030,2008-10-12,2008,ANA,ARI,6.0
1,2008020048,2008-10-15,2008,ANA,EDM,5.0
2,2008020061,2008-10-17,2008,ANA,S.J,4.0
3,2008020077,2008-10-19,2008,ANA,CAR,4.0
4,2008020135,2008-10-29,2008,ANA,DET,9.0


## 6) Clean master dataset (drop constant/duplicate columns)
Remove uninformative columns (all-NaN or constant) and exact duplicates to reduce noise and improve modeling stability.

In [14]:
# ============================================================
# 6) Clean master: drop constant/all-NaN + duplicate columns
# ============================================================
df = game_df.copy()

nunique = df.nunique(dropna=False)
constant_cols = nunique[nunique <= 1].index.tolist()

dup_mask = df.T.duplicated(keep="first")
duplicate_cols = df.columns[dup_mask].tolist()

manual_candidates = [
    "home_away_home", "home_away_away",
    "team_code_dup_home", "team_code_dup_away",
    "season_home", "season_away",
    "gameDate",
]
manual_to_drop = [c for c in manual_candidates if c in df.columns]

drop_cols = sorted(set(constant_cols) | set(duplicate_cols) | set(manual_to_drop))
df_clean = df.drop(columns=drop_cols, errors="ignore")

print("Dropped cols:", len(drop_cols))
print("Cleaned shape:", df_clean.shape)

MASTER_CLEAN_PATH = DATA_DIR / "master_cleaned.csv"
df_clean.to_csv(MASTER_CLEAN_PATH, index=False)
print("Saved:", MASTER_CLEAN_PATH.resolve())


Dropped cols: 67
Cleaned shape: (22118, 448)
Saved: /Users/thegootch/Desktop/Data Science/merrimack/capstone/capstone_project/data/master_cleaned.csv


## 7) EDA Figures (5)

**Figure mapping**
1. Total goals distribution
2. Total goals by season (mean + IQR)
3. Total goals vs xG environment (xG For + Against EWM fast)
4. Total goals vs rolling goal differential (EWM fast)
5. Correlation heatmap of key engineered features vs total goals

In [16]:
# ============================================================
# 7) Generate 5 EDA figures
# ============================================================
df = df_clean.copy()
df["game_date"] = pd.to_datetime(df["game_date"])

# -------- Figure 1: Total goals histogram --------
vals = df["total_goals"].dropna()
plt.figure(figsize=(9, 6))
plt.hist(vals, bins=range(int(vals.min()), int(vals.max()) + 2), edgecolor="white")
plt.title("Distribution of Total Goals (All Games)")
plt.xlabel("Total goals in game")
plt.ylabel("Number of games")
plt.axvline(vals.mean(), linestyle="--", linewidth=2, label=f"Mean = {vals.mean():.2f}")
plt.axvline(vals.median(), linestyle=":", linewidth=2, label=f"Median = {vals.median():.0f}")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_01_total_goals_hist.png", dpi=300)
plt.close()

# -------- Figure 2: Total goals by season --------
season_stats = (
    df.groupby("season")["total_goals"]
      .agg(mean="mean", q25=lambda x: np.percentile(x, 25), q75=lambda x: np.percentile(x, 75), n="count")
      .reset_index()
      .sort_values("season")
)
x = np.arange(len(season_stats))
plt.figure(figsize=(10, 6))
plt.plot(x, season_stats["mean"], marker="o")
plt.fill_between(x, season_stats["q25"], season_stats["q75"], alpha=0.2)
plt.title("Total Goals by Season (Mean with IQR Band)")
plt.xlabel("Season")
plt.ylabel("Total goals")
plt.xticks(x, season_stats["season"].astype(str), rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_02_total_goals_by_season.png", dpi=300)
plt.close()

# -------- Figure 3: Total goals vs xG environment --------
required = ["xGoalsFor_ewm_fast_home", "xGoalsAgainst_ewm_fast_home"]
if all(c in df.columns for c in required):
    df["xg_sum_ewm_fast_home"] = df["xGoalsFor_ewm_fast_home"] + df["xGoalsAgainst_ewm_fast_home"]
    plot_df = df[["xg_sum_ewm_fast_home", "total_goals"]].dropna()
    if len(plot_df) > 8000:
        plot_df = plot_df.sample(8000, random_state=RANDOM_SEED)

    plt.figure(figsize=(9, 6))
    plt.scatter(plot_df["xg_sum_ewm_fast_home"], plot_df["total_goals"], alpha=0.25)
    plt.title("Total Goals vs Rolling xG Environment (Home Team EWM Fast)")
    plt.xlabel("xGoalsFor_ewm_fast_home + xGoalsAgainst_ewm_fast_home")
    plt.ylabel("Total goals")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "eda_03_total_goals_vs_xg_sum.png", dpi=300)
    plt.close()
else:
    print("Skipping Figure 3: missing required columns:", required)

# -------- Figure 4: Total goals vs goal differential EWM fast --------
if "goal_diff_ewm_fast_home" in df.columns:
    plot_df = df[["goal_diff_ewm_fast_home", "total_goals"]].dropna()
    if len(plot_df) > 8000:
        plot_df = plot_df.sample(8000, random_state=RANDOM_SEED)

    plt.figure(figsize=(9, 6))
    plt.scatter(plot_df["goal_diff_ewm_fast_home"], plot_df["total_goals"], alpha=0.25)
    plt.title("Total Goals vs Rolling Goal Differential (Home Team EWM Fast)")
    plt.xlabel("goal_diff_ewm_fast_home")
    plt.ylabel("Total goals")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "eda_04_total_goals_vs_goal_diff_ewm.png", dpi=300)
    plt.close()
else:
    print("Skipping Figure 4: missing goal_diff_ewm_fast_home")

# -------- Figure 5: Correlation heatmap (matplotlib) --------
candidate_cols = [
    "total_goals",
    "goalsFor_ewm_fast_home",
    "goalsAgainst_ewm_fast_home",
    "highDangerShotsAgainst_home",
    "xGoalsAgainst_ewm_fast_home",
    "xGoalsFor_ewm_fast_home",
    "shotAttemptsFor_ewm_fast_home",
    "corsiPercentage_ewm_fast_home",
]
cols_present = [c for c in candidate_cols if c in df.columns]
corr_df = df[cols_present].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
plt.imshow(corr_df.values, aspect="auto")
plt.xticks(range(len(cols_present)), cols_present, rotation=90)
plt.yticks(range(len(cols_present)), cols_present)
plt.title("Correlation Heatmap (Selected Features vs Total Goals)")
plt.colorbar()
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_05_corr_heatmap_key_features.png", dpi=300)
plt.close()

print("Saved EDA figures to:", FIG_DIR.resolve())


Saved EDA figures to: /Users/thegootch/Desktop/Data Science/merrimack/capstone/capstone_project/figures


## 8) Modeling: Linear Regression baseline
Use a time-aware split (train first 80%, test last 20%) and TimeSeriesSplit cross-validation to control overfitting.

Also compute assumption diagnostics:
- Durbin–Watson (autocorrelation check)
- Breusch–Pagan (heteroskedasticity test)
And save residual diagnostic plots.

In [18]:
# ============================================================
# 8) Modeling — Linear Regression baseline
# ============================================================
df = df_clean.sort_values("game_date").reset_index(drop=True)

y = df["total_goals"].astype(float)

# numeric predictors only
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# leakage removal (targets and direct goal components)
leak_cols = [
    "total_goals",
    "home_goals", "away_goals", "home_win", "home_win_margin",
    "goalsFor_home", "goalsAgainst_home", "goalsFor_away", "goalsAgainst_away",
]
X = df[numeric_cols].drop(columns=[c for c in leak_cols if c in numeric_cols], errors="ignore")

# drop all-NaN
X = X.drop(columns=X.columns[X.isna().all()].tolist(), errors="ignore")

# time-aware split
split_index = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# impute
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# fit LR
lr = LinearRegression()
lr.fit(X_train_imp, y_train)

pred_train = lr.predict(X_train_imp)
pred_test = lr.predict(X_test_imp)

def eval_reg(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred, squared=False),
        "r2": r2_score(y_true, y_pred),
    }

lr_train_metrics = eval_reg(y_train, pred_train)
lr_test_metrics = eval_reg(y_test, pred_test)

print("Linear Regression performance")
print("Train:", lr_train_metrics)
print("Test :", lr_test_metrics)

# TimeSeries CV MAE (train only)
tscv = TimeSeriesSplit(n_splits=5)
cv_maes = []
y_train_np = y_train.to_numpy()

for tr_idx, va_idx in tscv.split(X_train_imp):
    m = LinearRegression()
    m.fit(X_train_imp[tr_idx], y_train_np[tr_idx])
    p = m.predict(X_train_imp[va_idx])
    cv_maes.append(mean_absolute_error(y_train_np[va_idx], p))

print("TimeSeries CV MAE scores:", cv_maes)
print("Mean CV MAE:", float(np.mean(cv_maes)))

# diagnostics
resid = (y_test.to_numpy() - pred_test)
dw = durbin_watson(resid)
bp = het_breuschpagan(resid, sm.add_constant(pred_test))

print("Durbin-Watson:", float(dw))
print("Breusch-Pagan (LM, LM p, F, F p):", [float(x) for x in bp])

# residual diagnostics figure
fig = plt.figure(figsize=(18, 5))

ax1 = fig.add_subplot(1, 3, 1)
ax1.scatter(pred_test, resid, alpha=0.3)
ax1.axhline(0, linestyle="--")
ax1.set_title("LR Residuals vs Predicted")
ax1.set_xlabel("Predicted total_goals")
ax1.set_ylabel("Residuals")

ax2 = fig.add_subplot(1, 3, 2)
sm.qqplot(resid, line="45", ax=ax2)
ax2.set_title("LR QQ Plot")

ax3 = fig.add_subplot(1, 3, 3)
ax3.hist(resid, bins=30, edgecolor="white")
ax3.set_title("LR Residual Distribution")
ax3.set_xlabel("Residual")
ax3.set_ylabel("Count")

plt.tight_layout()
plt.savefig(FIG_DIR / "lr_residual_diagnostics.png", dpi=300)
plt.close()

print("Saved:", (FIG_DIR / "lr_residual_diagnostics.png").resolve())


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Linear Regression performance
Train: {'mae': 0.5787698415037945, 'rmse': 0.7301880114790282, 'r2': 0.8985515818267475}
Test : {'mae': 0.5994106005061055, 'rmse': 0.7503149565218492, 'r2': 0.8948338614969389}
TimeSeries CV MAE scores: [0.6157879985929706, 0.5772557733476411, 0.5814634322432614, 0.5892987420288448, 0.6183912619088721]
Mean CV MAE: 0.5964394416243181
Durbin-Watson: 1.8646919043433368
Breusch-Pagan (LM, LM p, F, F p): [15.914158813490968, 6.628102100396583e-05, 15.964391985232888, 6.559531386032328e-05]
Saved: /Users/thegootch/Desktop/Data Science/merrimack/capstone/capstone_project/figures/lr_residual_diagnostics.png


## 9) Modeling: Tuned XGBoost
Train a tuned XGBoost regressor and evaluate it using the same time-aware split and TimeSeries CV. Also generate:
- Residual diagnostics plot
- Top-20 feature importances plot
- MAE train/test comparison plot (LR vs XGB)

In [20]:
# ============================================================
# 9) Modeling — Tuned XGBoost
# ============================================================
xgb = XGBRegressor(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=1.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=RANDOM_SEED,
    n_jobs=-1
)

xgb.fit(X_train_imp, y_train)

pred_train_xgb = xgb.predict(X_train_imp)
pred_test_xgb = xgb.predict(X_test_imp)

xgb_train_metrics = eval_reg(y_train, pred_train_xgb)
xgb_test_metrics = eval_reg(y_test, pred_test_xgb)

print("XGBoost (tuned) performance")
print("Train:", xgb_train_metrics)
print("Test :", xgb_test_metrics)

# TimeSeries CV MAE (train only)
cv_maes_xgb = []
for tr_idx, va_idx in tscv.split(X_train_imp):
    m = XGBRegressor(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=1.0,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
    m.fit(X_train_imp[tr_idx], y_train_np[tr_idx])
    p = m.predict(X_train_imp[va_idx])
    cv_maes_xgb.append(mean_absolute_error(y_train_np[va_idx], p))

print("TimeSeries CV MAE scores:", cv_maes_xgb)
print("Mean CV MAE:", float(np.mean(cv_maes_xgb)))

# diagnostics
resid_xgb = (y_test.to_numpy() - pred_test_xgb)
dw_xgb = durbin_watson(resid_xgb)
bp_xgb = het_breuschpagan(resid_xgb, sm.add_constant(pred_test_xgb))

print("Durbin-Watson:", float(dw_xgb))
print("Breusch-Pagan (LM, LM p, F, F p):", [float(x) for x in bp_xgb])

# residual diagnostics figure
fig = plt.figure(figsize=(18, 5))

ax1 = fig.add_subplot(1, 3, 1)
ax1.scatter(pred_test_xgb, resid_xgb, alpha=0.3)
ax1.axhline(0, linestyle="--")
ax1.set_title("XGB Residuals vs Predicted")
ax1.set_xlabel("Predicted total_goals")
ax1.set_ylabel("Residuals")

ax2 = fig.add_subplot(1, 3, 2)
sm.qqplot(resid_xgb, line="45", ax=ax2)
ax2.set_title("XGB QQ Plot")

ax3 = fig.add_subplot(1, 3, 3)
ax3.hist(resid_xgb, bins=30, edgecolor="white")
ax3.set_title("XGB Residual Distribution")
ax3.set_xlabel("Residual")
ax3.set_ylabel("Count")

plt.tight_layout()
plt.savefig(FIG_DIR / "xgb_residual_diagnostics.png", dpi=300)
plt.close()

print("Saved:", (FIG_DIR / "xgb_residual_diagnostics.png").resolve())

# Feature importance top 20
feat_imp = pd.DataFrame({"feature": X.columns, "importance": xgb.feature_importances_}).sort_values(
    "importance", ascending=False
).head(20)

plt.figure(figsize=(9, 10))
plt.barh(feat_imp["feature"][::-1], feat_imp["importance"][::-1])
plt.title("Top 20 Feature Importances (XGBoost)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(FIG_DIR / "xgb_feature_importance_top20.png", dpi=300)
plt.close()

print("Saved:", (FIG_DIR / "xgb_feature_importance_top20.png").resolve())

# LR vs XGB MAE comparison (train vs test)
plt.figure(figsize=(8, 6))
models = ["Linear Regression", "XGBoost (tuned)"]
train_maes = [lr_train_metrics["mae"], xgb_train_metrics["mae"]]
test_maes = [lr_test_metrics["mae"], xgb_test_metrics["mae"]]

xpos = np.arange(len(models))
width = 0.35
plt.bar(xpos - width/2, train_maes, width, label="Train MAE")
plt.bar(xpos + width/2, test_maes, width, label="Test MAE")
plt.xticks(xpos, models)
plt.ylabel("MAE")
plt.title("Model Comparison: MAE (Train vs Test)")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "model_mae_comparison.png", dpi=300)
plt.close()

print("Saved:", (FIG_DIR / "model_mae_comparison.png").resolve())


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


XGBoost (tuned) performance
Train: {'mae': 0.6877036278929914, 'rmse': 0.8628434174921873, 'r2': 0.8583424051579502}
Test : {'mae': 0.8467331246269115, 'rmse': 1.0614863808925414, 'r2': 0.7895167342516155}
TimeSeries CV MAE scores: [0.9015777415645531, 0.8203561513372986, 0.8030766419897083, 0.8555592550144797, 0.8636483612728345]
Mean CV MAE: 0.8488436302357748
Durbin-Watson: 1.9339033611164946
Breusch-Pagan (LM, LM p, F, F p): [114.06686796938646, 1.2598745951053342e-26, 117.03283431754264, 6.12238858432716e-27]
Saved: /Users/thegootch/Desktop/Data Science/merrimack/capstone/capstone_project/figures/xgb_residual_diagnostics.png
Saved: /Users/thegootch/Desktop/Data Science/merrimack/capstone/capstone_project/figures/xgb_feature_importance_top20.png
Saved: /Users/thegootch/Desktop/Data Science/merrimack/capstone/capstone_project/figures/model_mae_comparison.png


## 10) Unsupervised: Team style clustering (k=2)
Construct the team-level style matrix (38 teams × 37 features), standardize features, and fit KMeans with k=2.

Outputs:
- `figures/team_clusters_pca.png`
- `figures/team_clusters_cluster_means_heatmap.png`
- `figures/team_clusters_silhouette_scores.png`
- `artifacts/team_clusters_k2.csv`

In [22]:
# ============================================================
# 10) Team style clustering (k=2)
# ============================================================
STYLE_COLS_37 = [
    'goalsFor_ewm_fast_home', 'goalsFor_ewm_slow_home',
    'goalsAgainst_ewm_fast_home', 'goalsAgainst_ewm_slow_home',
    'goal_diff_ewm_fast_home', 'goal_diff_ewm_slow_home',
    'win_ewm_fast_home', 'win_ewm_slow_home',
    'xGoalsFor_ewm_fast_home', 'xGoalsFor_ewm_slow_home',
    'xGoalsAgainst_ewm_fast_home', 'xGoalsAgainst_ewm_slow_home',
    'shotsOnGoalFor_ewm_fast_home', 'shotsOnGoalFor_ewm_slow_home',
    'shotsOnGoalAgainst_ewm_fast_home', 'shotsOnGoalAgainst_ewm_slow_home',
    'shotAttemptsFor_ewm_fast_home', 'shotAttemptsFor_ewm_slow_home',
    'shotAttemptsAgainst_ewm_fast_home', 'shotAttemptsAgainst_ewm_slow_home',
    'totalShotCreditFor_ewm_fast_home', 'totalShotCreditFor_ewm_slow_home',
    'totalShotCreditAgainst_ewm_fast_home', 'totalShotCreditAgainst_ewm_slow_home',
    'xGoalsPercentage_ewm_fast_home', 'xGoalsPercentage_ewm_slow_home',
    'corsiPercentage_ewm_fast_home', 'corsiPercentage_ewm_slow_home',
    'fenwickPercentage_ewm_fast_home', 'fenwickPercentage_ewm_slow_home',
    'xGoalsPercentage_trend_7v30_home', 'corsiPercentage_trend_7v30_home',
    'fenwickPercentage_trend_7v30_home',
    'highDangerShotsFor_home', 'highDangerShotsAgainst_home',
    'goal_diff_away', 'goal_diff_ewm_fast_away',
]

TEAM_COL = "team_code_home"
missing = [c for c in [TEAM_COL] + STYLE_COLS_37 if c not in df_clean.columns]
if missing:
    raise ValueError("Missing required columns for clustering: " + str(missing))

team_style = df_clean[[TEAM_COL] + STYLE_COLS_37].groupby(TEAM_COL).mean(numeric_only=True)
print("Team style matrix shape:", team_style.shape)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(team_style)

# k=2 clustering
k = 2
kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init="auto")
clusters = kmeans.fit_predict(X_scaled)

# PCA scatter
pca = PCA(n_components=2, random_state=RANDOM_SEED)
coords = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame({"team": team_style.index, "PC1": coords[:, 0], "PC2": coords[:, 1], "cluster": clusters})

plt.figure(figsize=(10, 6))
for cl in sorted(pca_df["cluster"].unique()):
    sub = pca_df[pca_df["cluster"] == cl]
    plt.scatter(sub["PC1"], sub["PC2"], label=f"Cluster {cl}", alpha=0.9)
for _, r in pca_df.iterrows():
    plt.text(r["PC1"], r["PC2"], r["team"], fontsize=8)
plt.title("Team Style Clusters (k=2) in PCA Space")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "team_clusters_pca.png", dpi=300)
plt.close()

# cluster means heatmap (matplotlib)
team_style_with_cluster = team_style.copy()
team_style_with_cluster["cluster"] = clusters
cluster_means = team_style_with_cluster.groupby("cluster").mean(numeric_only=True)

plt.figure(figsize=(12, 6))
plt.imshow(cluster_means.values, aspect="auto")
plt.yticks(range(cluster_means.shape[0]), [f"Cluster {i}" for i in cluster_means.index])
plt.xticks(range(cluster_means.shape[1]), cluster_means.columns, rotation=90)
plt.title("Cluster Mean Style Features (k=2)")
plt.colorbar()
plt.tight_layout()
plt.savefig(FIG_DIR / "team_clusters_cluster_means_heatmap.png", dpi=300)
plt.close()

# silhouette scores by k
from sklearn.metrics import silhouette_score
sil = {}
for kk in range(2, 9):
    km = KMeans(n_clusters=kk, random_state=RANDOM_SEED, n_init="auto")
    lab = km.fit_predict(X_scaled)
    sil[kk] = float(silhouette_score(X_scaled, lab))

plt.figure(figsize=(8, 5))
ks = sorted(sil.keys())
plt.plot(ks, [sil[k] for k in ks], marker="o")
plt.title("Silhouette Scores by k (Team Style Clustering)")
plt.xlabel("k")
plt.ylabel("Silhouette score")
plt.tight_layout()
plt.savefig(FIG_DIR / "team_clusters_silhouette_scores.png", dpi=300)
plt.close()

# Save team -> cluster map
team_cluster_map = pca_df[["team", "cluster"]].sort_values(["cluster", "team"])
team_cluster_map.to_csv(ART_DIR / "team_clusters_k2.csv", index=False)

print("Saved clustering figures + team_clusters_k2.csv")
team_cluster_map.head(10)


Team style matrix shape: (38, 37)
Saved clustering figures + team_clusters_k2.csv


,team,cluster
0,ANA,0
1,ARI,0
2,ATL,0
4,BUF,0
6,CBJ,0
12,EDM,0
17,MTL,0
21,NYI,0
23,OTT,0
27,SEA,0


## 11) Save summary metrics for final report


In [24]:
# ============================================================
# 11) Save metrics.json 
# ============================================================
eda_stats = {
    "total_goals_mean": float(df_clean["total_goals"].mean()),
    "total_goals_median": float(df_clean["total_goals"].median()),
    "pct_total_goals_4_7": float(((df_clean["total_goals"] >= 4) & (df_clean["total_goals"] <= 7)).mean()),
    "pct_total_goals_8_plus": float((df_clean["total_goals"] >= 8).mean()),
    "season_low": str(df_clean.groupby("season")["total_goals"].mean().idxmin()),
    "season_low_mean": float(df_clean.groupby("season")["total_goals"].mean().min()),
    "season_high": str(df_clean.groupby("season")["total_goals"].mean().idxmax()),
    "season_high_mean": float(df_clean.groupby("season")["total_goals"].mean().max()),
    "season_high_minus_low": float(df_clean.groupby("season")["total_goals"].mean().max() - df_clean.groupby("season")["total_goals"].mean().min()),
}

# correlations used in report
if "xGoalsFor_ewm_fast_home" in df_clean.columns and "xGoalsAgainst_ewm_fast_home" in df_clean.columns:
    xg_sum = df_clean["xGoalsFor_ewm_fast_home"] + df_clean["xGoalsAgainst_ewm_fast_home"]
    eda_stats["corr_total_goals_xg_sum_ewm_fast_home"] = float(xg_sum.corr(df_clean["total_goals"]))

if "goal_diff_ewm_fast_home" in df_clean.columns:
    eda_stats["corr_total_goals_goal_diff_ewm_fast_home"] = float(df_clean["goal_diff_ewm_fast_home"].corr(df_clean["total_goals"]))

model_stats = {
    "linear_regression": {
        "train": {k: float(v) for k, v in lr_train_metrics.items()},
        "test": {k: float(v) for k, v in lr_test_metrics.items()},
        "tscv_mae_scores": [float(x) for x in cv_maes],
        "tscv_mae_mean": float(np.mean(cv_maes)),
        "durbin_watson": float(dw),
        "breusch_pagan": [float(x) for x in bp],
    },
    "xgboost_tuned": {
        "train": {k: float(v) for k, v in xgb_train_metrics.items()},
        "test": {k: float(v) for k, v in xgb_test_metrics.items()},
        "tscv_mae_scores": [float(x) for x in cv_maes_xgb],
        "tscv_mae_mean": float(np.mean(cv_maes_xgb)),
        "durbin_watson": float(dw_xgb),
        "breusch_pagan": [float(x) for x in bp_xgb],
    },
    "generalization_gaps": {
        "lr_mae_gap": float(lr_test_metrics["mae"] - lr_train_metrics["mae"]),
        "xgb_mae_gap": float(xgb_test_metrics["mae"] - xgb_train_metrics["mae"]),
    },
}

cluster_stats = {
    "team_style_shape": [int(team_style.shape[0]), int(team_style.shape[1])],
    "silhouette_scores": sil,
}

metrics = {"eda": eda_stats, "models": model_stats, "clustering": cluster_stats}

metrics_path = ART_DIR / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved:", metrics_path.resolve())


Saved: /Users/thegootch/Desktop/Data Science/merrimack/capstone/capstone_project/artifacts/metrics.json
